In [1]:

import pandas as pd

df = pd.read_csv("spam.csv", encoding='latin-1')[['v1', 'v2']]
df.columns = ['label', 'message']

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [2]:
%pip install deep-translator tqdm

Note: you may need to restart the kernel to use updated packages.


In [11]:
from deep_translator import GoogleTranslator
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

# Load dataset
df = pd.read_csv("spam.csv", encoding='latin-1')[['v1', 'v2']]
df.columns = ['label', 'message']

# Take smaller sample
spam = df[df['label'] == 'spam'].sample(250)
ham = df[df['label'] == 'ham'].sample(250)

df = pd.concat([spam, ham])

# Translate function
def translate_text(text):
    try:
        return GoogleTranslator(source='en', target='hi').translate(text)
    except:
        return text

# Apply translation
df['hindi'] = df['message'].progress_apply(translate_text)

# Save
df.to_csv("hindi_spam.csv", index=False)

print("✅ Done!")

  0%|          | 0/500 [00:00<?, ?it/s]

100%|██████████| 500/500 [06:31<00:00,  1.28it/s]

✅ Done!


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# CLEAN DATA 
df = df.dropna(subset=['hindi'])
df['hindi'] = df['hindi'].astype(str)


df['combined'] = df['message'] + " " + df['hindi']
df['combined'] = df['combined'].str.lower()

# FEATURES
X = df['combined']
y = df['label'].map({'ham': 0, 'spam': 1})

# SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#VECTORIZER
vectorizer = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# MODEL
model = LogisticRegression(max_iter=200)
model.fit(X_train_vec, y_train)

#  EVALUATION 
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

Accuracy: 0.97

Report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97        54
           1       0.98      0.96      0.97        46

    accuracy                           0.97       100
   macro avg       0.97      0.97      0.97       100
weighted avg       0.97      0.97      0.97       100



In [13]:
def predict_spam(text):
    text = str(text).lower()
    text_vec = vectorizer.transform([text])
    prediction = model.predict(text_vec)[0]
    return "🚨 Spam" if prediction == 1 else "✅ Not Spam"

# Test cases
print(predict_spam("आपने ₹5000 जीत लिए हैं अभी क्लिक करें"))
print(predict_spam("free offer click now win money"))
print(predict_spam("कल क्लास है मत भूलना"))
print(predict_spam("meeting at 5 pm"))

🚨 Spam
🚨 Spam
✅ Not Spam
✅ Not Spam
